## 1. Imports

We add the project root to `sys.path` so we can import `dataset` and `embedder` (VAE, SelfAttention2D).
Then we pull in PyTorch, numpy, matplotlib, and the custom modules.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

from dataset import RecordingDataset
from embedder import VAE
from tqdm import tqdm

## 2. Hyper-parameters

Device selection (CUDA if available), batch size, number of epochs, learning rate, latent dimensionality, and input image size.
The latent dim determines the bottleneck width; larger values preserve more detail at the cost of regularisation.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
EPOCHS = 50
LR = 1e-4
LATENT_DIM = 128
IMG_SIZE = 256

## 3. Dataset

`RecordingDataset` in `mode='vae'` scans `../try/` for `obs_*.npy` files and returns individual frames as `(C, H, W)` tensors.
The DataLoader shuffles and batches them.

In [ ]:
dataset = RecordingDataset(data_dir="../try", game="doom", mode="vae")
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

## 4. Model

We instantiate a configurable VAE with:
- 4 encoder conv layers: `[32, 64, 128, 256]` channels
- Kernel size 4, stride 2 throughout
- Self-attention inserted after layers 2 and 3 (mirrored in the decoder)
- Sigmoid output activation (pixel range `[0, 1]`)

See `embedder/vae.py` for the full architecture and `embedder/attention.py` for the SelfAttention2D implementation.

In [ ]:
model = VAE(
    in_channels=3,
    latent_dim=LATENT_DIM,
    img_size=IMG_SIZE,
    encoder_channels=[32, 64, 128, 256],
    encoder_kernels=[4, 4, 4, 4],
    encoder_strides=[2, 2, 2, 2],
    attention_layers=[2, 3],
    num_attention_heads=4,
    final_activation="sigmoid",
)

## 5. Optimiser

AdamW with a fixed learning rate of `1e-4`. No scheduler is used; the loss naturally plateaus.

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR)

## 6. Training loop

Each epoch runs over the full dataset in train mode, then evaluates on the last batch in eval mode:
- The reconstruction grid (original vs. reconstructed) is saved to `weights/doom/val_epoch_NNN.png`.
- The model checkpoint is saved to `weights/doom_{epoch}/` via `save_pretrained` (config.json + model.safetensors).

The loss is the standard VAE objective: MSE reconstruction + KL divergence (beta-VAE with beta=1.0).

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    last_batch = None

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in pbar:
        x = batch.to(DEVICE)

        optimizer.zero_grad()
        recon_x, mu, logvar = model(x)
        loss, _, _ = model.loss_vae(recon_x, x, mu, logvar, beta=1.0)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        last_batch = x
        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(dataloader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")

    model.eval()
    with torch.no_grad():
        recon = model(last_batch)[0]
        n = min(4, len(last_batch))
        orig = last_batch[:n].cpu()
        recon_imgs = recon[:n].cpu()

        fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
        for i in range(n):
            axes[0, i].imshow(np.transpose(orig[i].numpy(), (1, 2, 0)))
            axes[0, i].axis("off")
            axes[1, i].imshow(np.transpose(recon_imgs[i].numpy(), (1, 2, 0)))
            axes[1, i].axis("off")
        axes[0, 0].set_ylabel("Original")
        axes[1, 0].set_ylabel("Reconstruction")
        plt.suptitle(f"Epoch {epoch+1}/{EPOCHS}")
        plt.tight_layout()
        plt.savefig(f"../weights/doom/val_epoch_{epoch+1:03d}.png", dpi=150)
        plt.close()

    model.save_pretrained(f"../weights/doom_{epoch}")
    print(f"Checkpoint saved to ../weights/doom_{epoch}")

## 7. Inspect results

After training completes (or is interrupted), load the last checkpoint and visualise a few reconstructions.
The saved `.png` grids under `weights/doom/` already show per-epoch progress.

In [ ]:
# Example: load a saved checkpoint and run a quick sanity check
# model = VAE.from_pretrained("../weights/doom_49", map_location=DEVICE)
# model.eval()
# ...